In [2]:
!pip install xlrd

  Obtaining dependency information for xlrd from https://files.pythonhosted.org/packages/a6/0c/c2a72d51fe56e08a08acc85d13013558a2d793028ae7385448a6ccdfae64/xlrd-2.0.1-py2.py3-none-any.whl.metadata
   ---------------------------------------- 0.0/96.5 kB ? eta -:--:--
   ----------------------------- ---------- 71.7/96.5 kB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 96.5/96.5 kB 2.8 MB/s eta 0:00:00


In [3]:
# Importing Python Libraries
import pandas as pd
import xlrd
from datetime import date
from openpyxl import load_workbook
import re
import shutil

In [4]:
# Converting old xlsx excel format of STIBO Extract to Dictionary
def xlsx_to_dict(filePath):

  df = pd.read_excel(filePath, engine='openpyxl')

  result_dict = df.to_dict(orient='records')

  return result_dict


In [5]:
# Picking up the STIBO extract from the dedicated file
#filePath = r'C:\Users\SYSTCS491\Documents\Monday CPCR\CPCR Primary Extract.xlsx'
filePath = r'C:\Users\Admin\Documents\Udemy\Microsoft Azure Data Engineer\Poratl_Automation\CPCR Primary Extract.xlsx'
            
data_dict = xlsx_to_dict(filePath)


In [6]:
# Declaring the necessary list variables to store dictionary in structured format for ease of operations
agreeItemNoList = []
costStatusList = []
agreeItemNetCostList = []
agreeItemProposedCostList = []
agreeCaseCountList = []


In [9]:
# df = pd.read_excel(filePath, engine='openpyxl')
# df

In [10]:
# Manual Operation Part 1
# Checking and sorting out Data
for item in data_dict:
  if item['AttributeID'] == 'AgreementItemNumber':
    agreeItemNoList.append(item)
  elif item['AttributeID'] == 'AgreementItemNetCostPrice':
    agreeItemNetCostList.append(item)
  elif item['AttributeID'] == 'ItemProposedCostPrice':
    agreeItemProposedCostList.append(item)
  elif item['AttributeID'] == 'AgreementItemCaseCount':
    agreeCaseCountList.append(item)
  elif item['AttributeID'] == 'CostStatus':
    costStatusList.append(item)

print("Agreement Item Number List",len(agreeItemNoList))
print("Agreement Net Cost",len(agreeItemNetCostList))
print("Agreement Proposed Cost",len(agreeItemProposedCostList))
print("Agreement Case Count",len(agreeCaseCountList))
print("Agreement Cost Status",len(costStatusList))


Agreement Item Number List 3
Agreement Net Cost 3
Agreement Proposed Cost 3
Agreement Case Count 3
Agreement Cost Status 3


In [13]:
# Declaring Mast List for saving the first manual operation Final Result
MasterList = []

# V Lookup 1st Stage Automated Process
for item in agreeItemNoList:
  #mergedName = item['ID'] + '-' + item['ProductID']
  mergedName = str(item['ID']) + '-' + item['ProductID']

  NewDict = {}
  currentCost = 'NA'
  proposedCost = 'NA'
  caseCount = 'NA'
  costStatus = 'NA'
  NewDict['ID'] = item['ID']
  NewDict['Item Supplier Site'] = item['ProductID']
  NewDict['Product ID'] = item['Value']

  for item2 in agreeItemNetCostList:
    #if mergedName == item2['ID'] + '-' + item2['ProductID']:
    if mergedName == str(item2['ID']) + '-' + item2['ProductID']:
      currentCost = item2['Value']
      break

  for item3 in agreeItemProposedCostList:
    #if mergedName == item3['ID'] + '-' + item3['ProductID']:
    if mergedName == str(item3['ID']) + '-' + item3['ProductID']:    
      proposedCost = item3['Value']
      break

  for item4 in agreeCaseCountList:
    #if mergedName == item4['ID'] + '-' + item4['ProductID']:
    if mergedName == str(item4['ID']) + '-' + item4['ProductID']:    
      caseCount = item4['Value']
      break

  for item5 in costStatusList:
    #if mergedName == item5['ID'] + '-' + item5['ProductID']:
    if mergedName == str(item5['ID']) + '-' + item5['ProductID']:    
      costStatus = item5['Value']
      break

  NewDict['Current Cost Price'] = currentCost
  NewDict['Proposed Cost Price'] = proposedCost
  NewDict['Case Count'] = caseCount
  NewDict['Cost Status'] = costStatus
  MasterList.append(NewDict)

In [14]:
# Count Validation
print("Number of Objects in Pre Sheet is: ",len(MasterList))


Number of Objects in Pre Sheet is:  3


In [15]:
# Declaring pattern Detection to remove Failed Agreements
count = 0
pattern = r"\bFailed\b"

for item in MasterList[:]:
  if re.search(pattern, item['Cost Status']):
    MasterList.remove(item)
    count = count + 1
print("Number of Items with Failed is: ",count)
print("Number of Items in Pre Sheet when Failed Items Deleted is: ",len(MasterList))


Number of Items with Failed is:  1
Number of Items in Pre Sheet when Failed Items Deleted is:  2


In [16]:
# Completion of step One and creation of Pre Sheet for later reference. Conclusion of 1st Set of Operations.
df = pd.DataFrame(MasterList)
df.to_excel('Pre-Sheet.xlsx', index=False)

In [19]:
df

,ID,Item Supplier Site,Product ID,Current Cost Price,Proposed Cost Price,Case Count,Cost Status
0,1,A1,Item1,100,110,5,Active
1,3,A3,Item3,150,160,8,Active


In [35]:
# Manual Operation Part 2
# Importing the Secondary Extract from STIBO into the code
#filePath = r'C:\Users\SYSTCS491\Documents\Monday CPCR\CPCR Secondary Extract.xls'
filePath = r'C:\Users\Admin\Documents\Udemy\Microsoft Azure Data Engineer\Poratl_Automation\CPCR Secondary Extract.xls'

In [36]:
#Converting and opening the Workbook as Dictionary for ease of processing
workbook = xlrd.open_workbook(filePath)
sheet = workbook.sheet_by_index(0)

data = []

header = sheet.row_values(0)

for row in range(1, sheet.nrows):
    row_data = sheet.row_values(row)
    data.append(dict(zip(header, row_data)))

print("The Total Number of Items in the Secondary Extracted Excel from STEP is: ",len(data))


The Total Number of Items in the Secondary Extracted Excel from STEP is:  3


In [37]:
# Declaring the necessary variables that will be used for final V Lookup operation
FinalList = []
TradingTeam = None
TraderTeam = None
Status = None
CostPriceStartDate = None
SupplierNumber = None
Supplier = None
STEPSupplierTradingUnitID = None
SupplierTradingUnitName= None
SupplierTradingUnitRMSID= None
CPCRCreationDate = None


In [39]:
# Commencing the final V Look up Manual Operation from secondary sheet
for item in MasterList:
  for attr in data:
    if item['ID'] == attr['<ID>']:
      TradingTeam = attr['Trading Team']
      TraderTeam = attr['Trader Team']
      Status = attr['Status']
      CostPriceStartDate = attr['Cost Price Start Date']
      SupplierNumber = attr['RMS Supplier ID']
      Supplier = attr['Supplier']
      STEPSupplierTradingUnitID = attr['STEP Supplier Trading Unit ID']
      SupplierTradingUnitName= attr['Supplier Trading Unit Name']
      SupplierTradingUnitRMSID= attr['RMS Supplier Trading Unit ID']
    
      #######
      CPCRCreationTimeStamp = attr.get('Complete Cost Price Change Request - On Entry TS', None)
      if isinstance(CPCRCreationTimeStamp, str):
        CPCRCreationTimeStamp = CPCRCreationTimeStamp[0:10]
      elif isinstance(CPCRCreationTimeStamp, float):
        CPCRCreationTimeStamp = None # Handle NaN or unexpected float values
      break      
    
      #######  
#        CPCRCreationTimeStamp = attr['Complete Cost Price Change Request - On Entry TS'][0:10]
#        break

  item['Trading Team'] = TradingTeam
  item['Trader Team'] = TraderTeam
  item['Status'] = Status
  item['Cost Price Start Date'] = CostPriceStartDate
  item['Supplier Number'] = SupplierNumber
  item['Supplier'] = Supplier
  item['STEP Supplier Trading Unit ID'] = STEPSupplierTradingUnitID
  item['Supplier Trading Unit Name'] = SupplierTradingUnitName
  item['Supplier Trading Unit RMS ID'] = SupplierTradingUnitRMSID
  item['CPCR Entry Date'] = CPCRCreationTimeStamp
  FinalList.append(item)

print("The Length of the Final Sheet before Deleting Data is: ", len(FinalList))


The Length of the Final Sheet before Deleting Data is:  2


In [40]:
# Manual Operation Part 3
# Removing Cost Status Column and moving the two columns Case Count and CPCR Entry Date to the last
popV = 'Case Count'
popC = 'CPCR Entry Date'
for item in FinalList:
  item['Case Count'] = item.pop(popV)
  item['CPCR Entry Date'] = item.pop(popC)
  item.pop('Cost Status')


In [41]:
# Removing the Objects with Status as Closed (Not Accepted) and Rejected
count = 0
for item in FinalList[:]:
  if item['Status'] == 'Closed (Not Accepted)' or item['Status'] == 'Rejected':
    FinalList.remove(item)
    count = count + 1
print("Total Number of Objects having Closed (Not Accepted) and Rejected is: ", count)
print("The Total Number of Objects in the List after Removing Undesired Data is: ", len(FinalList))


Total Number of Objects having Closed (Not Accepted) and Rejected is:  0
The Total Number of Objects in the List after Removing Undesired Data is:  2


In [42]:
# Manual Operation Part 4
# Declaring the necessary List for segregating the Sheets in Final Report
FinalListInProgress = []
FinalListCompleted = []


In [43]:
# Segregating the Items based on Status to different Sheets
for item in FinalList:
  if item['Status'] == 'In Progress':
    FinalListInProgress.append(item)
  else:
    FinalListCompleted.append(item)

print("Total Number of Objects in the In Progress List is: ",len(FinalListInProgress))
print("Total Number of Objects in the Completed List is: ",len(FinalListCompleted))


Total Number of Objects in the In Progress List is:  1
Total Number of Objects in the Completed List is:  1


In [44]:
# Converting the Dictionary to dataframe format for conversion to Excel Document
df1 = pd.DataFrame(FinalListInProgress)
df2 = pd.DataFrame(FinalListCompleted)

df1['Cost Price Start Date'] = pd.to_datetime(df1['Cost Price Start Date']).dt.date
df2['Cost Price Start Date'] = pd.to_datetime(df2['Cost Price Start Date']).dt.date
df1['CPCR Entry Date'] = pd.to_datetime(df1['CPCR Entry Date']).dt.date
df2['CPCR Entry Date'] = pd.to_datetime(df2['CPCR Entry Date']).dt.date

today = str(date.today())

name = 'Cost Price Report -' + today + '.xlsx'

with pd.ExcelWriter(name) as writer:
  df2.to_excel(writer, sheet_name='Completed', index=False)
  df1.to_excel(writer, sheet_name='In Progress', index=False)
  

In [46]:
# Conversion to Excel and storing the final report in the dedicated File
#new_path = "C:/Users/SYSTCS491/Documents/Monday CPCR/" + name
new_path = "C:/Users/Admin/Documents/Udemy/Microsoft Azure Data Engineer/Poratl_Automation/Report/" + name


shutil.move(name, new_path)

print("Execution Completed !")

Execution Completed !
